In [ ]:
# COMPLETE ROUTE -> SENSOR -> SPEED HISTORY TEST

import os
import pickle
import numpy as np
import pandas as pd



DATA_PATH = '../../dataset/METR-LA'

GRAPH_PATH = os.path.join(
    DATA_PATH,
    'adj_mx_METR-LA.pkl'
)

SENSOR_PATH = os.path.join(
    DATA_PATH,
    'sensor_locations.csv'
)



print('=' * 70)
print('1. LOADING SENSOR LOCATIONS')
print('=' * 70)

sensors = pd.read_csv(
    SENSOR_PATH
)

sensors['sensor_id'] = (
    sensors['sensor_id']
    .astype(str)
)

print(
    'Sensors:',
    len(sensors)
)

print(
    sensors[
        [
            'sensor_id',
            'latitude',
            'longitude'
        ]
    ].head()
)



print()
print('=' * 70)
print('2. LOADING SENSOR INDEX MAPPING')
print('=' * 70)

with open(
    GRAPH_PATH,
    'rb'
) as f:

    adj_data = pickle.load(
        f,
        encoding='latin1'
    )


adj_sensor_ids = [
    str(x)
    for x in adj_data[0]
]


sensor_id_mapping = {
    sensor_id: idx
    for idx, sensor_id
    in enumerate(
        adj_sensor_ids
    )
}


print(
    'Total model sensors:',
    len(sensor_id_mapping)
)



print()
print('=' * 70)
print('3. SEARCHING FOR SPEED DATA')
print('=' * 70)

print()

for file in os.listdir(DATA_PATH):

    print(
        ' -',
        file
    )



candidate_files = []

for file in os.listdir(DATA_PATH):

    lower = file.lower()

    if (
        lower.endswith('.h5')
        or
        lower.endswith('.hdf')
        or
        lower.endswith('.csv')
        or
        lower.endswith('.npy')
        or
        lower.endswith('.npz')
    ):

        candidate_files.append(
            file
        )


print()
print(
    'Candidate data files:',
    candidate_files
)



speed_matrix = None


for file in candidate_files:

    path = os.path.join(
        DATA_PATH,
        file
    )

    try:

     
        if file.lower().endswith(
            ('.h5', '.hdf')
        ):

            df_speed = pd.read_hdf(
                path
            )

            arr = df_speed.to_numpy(
                dtype=np.float32
            )

            if (
                arr.ndim == 2
                and
                arr.shape[1] == 207
            ):

                speed_matrix = arr

                print(
                    'Loaded speed data:',
                    file
                )

                break


        elif file.lower().endswith(
            '.npy'
        ):

            arr = np.load(
                path,
                mmap_mode='r'
            )

            if (
                arr.ndim == 2
                and
                arr.shape[1] == 207
            ):

                speed_matrix = np.asarray(
                    arr,
                    dtype=np.float32
                )

                print(
                    'Loaded speed data:',
                    file
                )

                break


        elif file.lower().endswith(
            '.npz'
        ):

            data = np.load(
                path
            )

            for key in data.files:

                arr = data[key]

                if (
                    arr.ndim == 2
                    and
                    arr.shape[1] == 207
                ):

                    speed_matrix = np.asarray(
                        arr,
                        dtype=np.float32
                    )

                    print(
                        'Loaded speed data:',
                        file,
                        'key:',
                        key
                    )

                    break

            if speed_matrix is not None:
                break


       
        elif file.lower().endswith(
            '.csv'
        ):

            df_speed = pd.read_csv(
                path
            )


            numeric = (
                df_speed
                .select_dtypes(
                    include=[np.number]
                )
            )

            if numeric.shape[1] == 207:

                speed_matrix = numeric.to_numpy(
                    dtype=np.float32
                )

                print(
                    'Loaded speed data:',
                    file
                )

                break


    except Exception as e:

        print(
            f'Skipped {file}: {e}'
        )



print()
print('=' * 70)
print('4. SPEED MATRIX CHECK')
print('=' * 70)

if speed_matrix is None:

    print(
        '❌ Could not automatically find '
        'the 207-sensor speed matrix.'
    )

    print()
    print(
        'Do NOT continue yet.'
    )

else:

    print(
        'Speed matrix shape:',
        speed_matrix.shape
    )

    print(
        'dtype:',
        speed_matrix.dtype
    )

    print(
        'NaN:',
        np.isnan(
            speed_matrix
        ).sum()
    )

    print(
        'Inf:',
        np.isinf(
            speed_matrix
        ).sum()
    )


print()
print('=' * 70)
print('5. ROUTE SENSOR → MODEL INDEX')
print('=' * 70)


route_sensor_ids = [
    '773024',
    '764853'
]


for sensor_id in route_sensor_ids:

    if sensor_id in sensor_id_mapping:

        idx = sensor_id_mapping[
            sensor_id
        ]

        print(
            f'{sensor_id} → index {idx}'
        )

    else:

        print(
            f'{sensor_id} ❌ NOT FOUND'
        )


print()
print('=' * 70)
print('6. HISTORICAL SPEED TEST')
print('=' * 70)


if speed_matrix is not None:

    test_time = 1000

    for sensor_id in route_sensor_ids:

        if sensor_id not in sensor_id_mapping:
            continue

        sensor_idx = (
            sensor_id_mapping[
                sensor_id
            ]
        )

        speed = float(
            speed_matrix[
                test_time,
                sensor_idx
            ]
        )

        print(
            f'Sensor {sensor_id} | '
            f'index={sensor_idx} | '
            f't={test_time} | '
            f'speed={speed:.2f}'
        )

print()
print('=' * 70)
print('7. HISTORY WINDOW TEST')
print('=' * 70)


if speed_matrix is not None:

    test_time = 1000

    history_steps = 12


    for sensor_id in route_sensor_ids:

        if sensor_id not in sensor_id_mapping:
            continue

        sensor_idx = (
            sensor_id_mapping[
                sensor_id
            ]
        )


        start = max(
            0,
            test_time - history_steps
        )


        history = speed_matrix[
            start:test_time + 1,
            sensor_idx
        ]


        print()

        print(
            f'Sensor {sensor_id}'
        )

        print(
            'Index:',
            sensor_idx
        )

        print(
            'History length:',
            len(history)
        )

        print(
            'History:',
            np.round(
                history,
                2
            )
        )



print()
print('=' * 70)
print('🏁 FINAL TEST STATUS')
print('=' * 70)


print(
    'Sensor mapping:'
    if len(sensor_id_mapping) == 207
    else: pass
)


print(
    'Route sensors:',
    route_sensor_ids
)


print(
    'Speed matrix:',
    ' READY'
    if speed_matrix is not None
    else ' NOT FOUND'
)


if speed_matrix is not None:

    print()
    print(
        'Route -> Sensor ->'
        'Model Index → Speed History '
        'pipeline is READY.'
    )

else:

    print()
    print(
        ' Only speed-data loading '
        'needs to be fixed.'
    )

1. LOADING SENSOR LOCATIONS
Sensors: 207
  sensor_id  latitude  longitude
0    773869  34.15497 -118.31829
1    767541  34.11621 -118.23799
2    767542  34.11641 -118.23819
3    717447  34.07248 -118.26772
4    717446  34.07142 -118.26572

2. LOADING SENSOR INDEX MAPPING
Total model sensors: 207

3. SEARCHING FOR SPEED DATA

 - adj_METR-LA.pkl
 - adj_mx_METR-LA.pkl
 - METR-LA.h5
 - sensor_locations.csv

Candidate data files: ['METR-LA.h5', 'sensor_locations.csv']
Loaded speed data: METR-LA.h5

4. SPEED MATRIX CHECK
Speed matrix shape: (34272, 207)
dtype: float32
NaN: 0
Inf: 0

5. ROUTE SENSOR → MODEL INDEX
773024 → index 93
764853 → index 144

6. HISTORICAL SPEED TEST
Sensor 773024 | index=93 | t=1000 | speed=65.62
Sensor 764853 | index=144 | t=1000 | speed=52.50

7. HISTORY WINDOW TEST

Sensor 773024
Index: 93
History length: 13
History: [67.   66.   67.   66.38 66.44 66.67 64.75 64.88 67.38 67.78 65.43 63.67
 65.62]

Sensor 764853
Index: 144
History length: 13
History: [50.89 56.38 5